# Fan review — one carved design, full 12-blade fan, folded + unfolded

Final visual review before printing. Pick ONE design (by wind rank), convert its **carved TO density**
into a printable watertight mesh (marching-cubes at ρ=0.5 — this is what you slice), assemble the
**12-blade fan**, and render it **folded** and **unfolded** (drag to rotate = all angles). Also writes
the printable **STL**. Everything is the carved geometry that actually prints — not a placeholder.

## 1. Connect Drive

In [ ]:
# Drive connect ONLY (kept separate from the repo/deps install below).
import importlib.util
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/fanopt")
else:
    DRIVE_ROOT = Path.cwd() / "data"
print("drive root:", DRIVE_ROOT)

## 2. Repo + deps (+ scikit-image for marching cubes)

In [ ]:
import importlib.util, os, subprocess, sys
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None
BRANCH = "main"  # the TO tool + this notebook land on main
REPO = Path("/content/fan-optimization") if IN_COLAB else Path.cwd()
if IN_COLAB:
    if not REPO.exists():
        subprocess.run(["git", "clone", "-b", BRANCH,
                        "https://github.com/clingergab/fan-optimization.git", str(REPO)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO), "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO), "pull", "origin", BRANCH], check=True)
    subprocess.run("apt-get install -qq -y libglu1-mesa libxrender1 libxcursor1 "
                   "libxft2 libxinerama1 unzip".split(), check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[bo]"], check=True)
    # TO stack: gmsh + CadQuery for the solid mesh, scikit-fem for the 3D FEA.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "gmsh", "cadquery", "scikit-fem"], check=True)
for p in (str(REPO), str(REPO / "src"), str(REPO / "scripts")):
    if p not in sys.path:
        sys.path.insert(0, p)
print("repo:", REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-image"], check=True)
print("scikit-image ready")


## 3. Config — pick ONE design + surface resolution

In [ ]:
SHARED_DIR   = DRIVE_ROOT / "campaign_trapezoid"
VERIFICATION = DRIVE_ROOT / "stage3_verify_blade" / "verification.json"
OUT_DIR      = DRIVE_ROOT / "stage4_blade_to"          # the TO carved density fields

SELECT_RANK  = 0        # which design to review: 0 = best wind (fine J_fan), 1 = 2nd, ...
TOP_K        = 4        # resolve the top-4 print candidates, then pick SELECT_RANK

# Deploy fan-out direction: -1 = top layers fan RIGHT / bottom fan LEFT (preferred - shingles the
# overlap toward the airflow to reduce leakage); +1 = the other way. Fold-symmetric, aero-neutral.
DEPLOY_DIRECTION = -1

# Marching-cubes voxel pitch. The TO cloud is ~0.6 mm, so 0.4-0.5 mm captures its real detail;
# going below ~0.5 mm mostly adds staircase facets (nearest-centroid upsampling), not new detail,
# and a heavier mesh. The marching-cubes surface sits at mesh resolution (~0.6 mm) - the smooth-CAD
# skin route is the refinement for a production-grade surface.
VOXEL_PITCH_M = 0.5e-3

# --- Deployed-gap tightening -------------------------------------------------------------------
# The deployed inter-blade gap floors at the FOLD CLEARANCE (the blades stack one boss-height apart
# on the pin). Shrinking it packs the deck tighter -> smaller gap. 0.3 mm is verified to STILL FOLD
# for the top-3 (penetration = -clearance <= 0); 0.2 mm folds geometrically but is a print-tolerance
# gamble. >>> PRINT ONE PAIR AND FOLD-TEST before trusting anything below 0.4 mm. <<<
FOLD_CLEARANCE_M = 0.3e-3          # default deck clearance is 0.4 mm; this is the tighter print deck

# NOTE on the click: a hub bump/recess detent was investigated and is GEOMETRICALLY INFEASIBLE on the
# 6 mm boss - at the 13.3-deg deploy pitch the folded->deployed arc travel is only ~0.8 mm, narrower
# than any printable detent feature, so recesses merge into a groove with no click. A real click needs
# the detent at large radius (blade rim, ~46 mm of travel) + a compliant spring; deferred as a decision.
assert OUT_DIR.exists(), f"Stage-4 output not found: {OUT_DIR}"
print(f"reviewing rank {SELECT_RANK} of top-{TOP_K} | voxel {VOXEL_PITCH_M*1e3:.2f} mm | "
      f"deck clearance {FOLD_CLEARANCE_M*1e3:.2f} mm")


## 4. Resolve the design + build its printable carved mesh

Loads the carved density, marching-cubes it to a watertight surface, and reports the blade size, the
mesh-enclosed mass (cross-check vs the ~20 g screen mass), and the **fold check** (folded 12-stack vs
the 90 mm cap) for this specific pick.

In [ ]:
import numpy as np
from fanopt.cfd.blade_verify import top_verified_designs
from fanopt.geometry.to_stl import carved_blade_mesh, write_binary_stl, laplacian_smooth
from fanopt.geometry.blade import folded_stack_height_m, fold_margin_m, MAX_FOLDED_STACK_HEIGHT_M, BLADE_COUNT, layer_spacing_m
from fanopt.geometry.blade_cad import fold_penetration_m
from fanopt.geometry.fan_seal import deployed_gap_range_m
from fanopt.geometry.schema import RHO_PETG_KG_PER_M3

designs = top_verified_designs(SHARED_DIR, VERIFICATION, top_k=TOP_K)  # sorted by fine J_fan (best first)
name, params, j3d = designs[SELECT_RANK]
dens = np.load(OUT_DIR / f"{name}_density.npy")
cen  = np.load(OUT_DIR / f"{name}_centroids.npy")
verts, faces = carved_blade_mesh(dens, cen, voxel_pitch_m=VOXEL_PITCH_M)

tris = verts[faces]
mesh_vol = abs(np.einsum('ij,ij->i', tris[:, 0], np.cross(tris[:, 1], tris[:, 2])).sum() / 6.0)
bbox_mm = (verts.max(0) - verts.min(0)) * 1e3
stack_mm = folded_stack_height_m(params) * 1e3
print(f"SELECT_RANK {SELECT_RANK} = best-wind #{SELECT_RANK} by FINE J_fan -> design {name}")
print(f"  (the NN_ prefix in the name is the ORIGINAL COARSE rank, not the wind rank)  J_fan_3d {j3d:.3e}")
print(f"carved mesh: {len(faces):,} triangles  |  bbox {np.round(bbox_mm,1)} mm")
print(f"one-blade mass (mesh volume): {mesh_vol*RHO_PETG_KG_PER_M3*1e3:.1f} g  ->  fan x{BLADE_COUNT} = "
      f"{mesh_vol*RHO_PETG_KG_PER_M3*BLADE_COUNT*1e3:.0f} g")
print(f"FOLD CHECK: folded 12-stack {stack_mm:.1f} mm vs {MAX_FOLDED_STACK_HEIGHT_M*1e3:.0f} mm cap  "
      f"-> margin {fold_margin_m(params)*1e3:+.1f} mm  ({'OK' if fold_margin_m(params) >= 0 else 'OVER'})")

# Smooth the CARVED marching-cubes surface a touch for a cleaner render (cosmetic; the STL uses the raw mesh).
verts_smooth = laplacian_smooth(verts, faces, iterations=10)

# --- Deployed gap: BEFORE (0.4 mm deck) vs AFTER (tighter FOLD_CLEARANCE_M) -------------------
# The deployed inter-blade slot = envelope + clearance - panel; its FLOOR is the fold clearance. So
# the gap can't reach zero in a z-stacked deck, but shrinking the clearance shrinks it. Both the max
# slot and the floor drop by the clearance reduction. The fold check confirms the tighter deck STILL
# NESTS (penetration <= 0) - if it ever prints positive, back the clearance off.
g0_lo, g0_hi = deployed_gap_range_m(params)                          # default 0.4 mm deck
g1_lo, g1_hi = deployed_gap_range_m(params, clearance_m=FOLD_CLEARANCE_M)  # tighter print deck
pen_mm = fold_penetration_m(params, clearance_m=FOLD_CLEARANCE_M) * 1e3
print(f"DEPLOYED GAP  before (0.40 mm deck): {g0_lo*1e3:.2f}-{g0_hi*1e3:.2f} mm  ->  "
      f"after ({FOLD_CLEARANCE_M*1e3:.2f} mm deck): {g1_lo*1e3:.2f}-{g1_hi*1e3:.2f} mm  "
      f"({(1-g1_hi/g0_hi)*100:.0f}% smaller)")
print(f"TIGHTER-DECK FOLD CHECK: penetration {pen_mm:+.2f} mm  ({'NESTS' if pen_mm <= 0 else 'COLLIDES - back off'})")
print(f"  reminder: gap FLOOR = fold clearance; PRINT-AND-FOLD-TEST before trusting < 0.4 mm.")


## 5. Export the printable STL — carved dish + reduced-clearance boss

The raw marching-cubes mesh bakes the boss at the 0.4 mm deck, so the tighter deck can't be retrofit
onto it. `carved_blade_with_boss` voids the hub column of the TO density (a hair INSIDE the boss OD so
the bodies genuinely overlap, not just abut), keeps the carved dish (the aero/structural surface TO
produced — untouched), then drops in a fresh CAD boss at `FOLD_CLEARANCE_M`. The overlapping watertight
bodies are emitted as one mesh; mainstream slicers union them into one connected part.

In [ ]:
from fanopt.geometry.blade_cad import carved_blade_with_boss

fused_v, fused_f = carved_blade_with_boss(dens, cen, params, voxel_pitch_m=VOXEL_PITCH_M,
                                          clearance_m=FOLD_CLEARANCE_M)
stl_path = OUT_DIR / f"rank{SELECT_RANK}_{name}_carved_c{FOLD_CLEARANCE_M*1e3:.1f}mm.stl"
write_binary_stl(fused_v, fused_f, stl_path)   # writes MILLIMETRES (metres x 1000) so slicers read true size
print(f"wrote {stl_path}")
print(f"  {(stl_path.stat().st_size)/1e6:.1f} MB | {len(fused_f):,} triangles | units MM | deck {FOLD_CLEARANCE_M*1e3:.2f} mm"
      f"  -> slice this (print ALL 12)")


## 6. Render the full 12-blade fan — folded + unfolded

Two interactive scenes: **folded** (stowed z-stack) and **unfolded** (deployed, ±73° fan). **Drag to
rotate for all angles**; each blade a distinct colour. This is the carved geometry that prints.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from fanopt.geometry.fan_pose import pose_fan

# Render the CARVED TO mesh (Laplacian-smoothed for a clean surface) - this is the actual printed
# geometry, NOT the uncarved BO CAD. Drop VOXEL_PITCH_M in the config for a finer carved surface.
rverts, rfaces = verts_smooth, faces

palette = (["#4e79a7", "#f28e2b", "#e15759", "#76b7b2", "#59a14f", "#edc948",
            "#b07aa1", "#ff9da7", "#9c755f", "#bab0ac", "#86bcb6", "#d37295"])

def fan_traces(deployed):
    out = []
    posed = pose_fan(rverts, rfaces, params, deployed=deployed, direction=DEPLOY_DIRECTION, clearance_m=FOLD_CLEARANCE_M)
    for i, (vv, ff) in enumerate(posed):
        out.append(go.Mesh3d(x=vv[:, 0], y=vv[:, 1], z=vv[:, 2],
                             i=ff[:, 0], j=ff[:, 1], k=ff[:, 2],
                             color=palette[i % 12], flatshading=True, opacity=1.0,
                             showscale=False, name=f"blade {i}"))
    return out

fig = make_subplots(rows=1, cols=2, specs=[[{"type": "scene"}, {"type": "scene"}]],
                    subplot_titles=[f"FOLDED (stowed)  —  {name[:11]}", "UNFOLDED (deployed)"])
for t in fan_traces(False): fig.add_trace(t, row=1, col=1)
for t in fan_traces(True):  fig.add_trace(t, row=1, col=2)
for sc in ("scene", "scene2"):
    fig.update_layout(**{sc: dict(aspectmode="data",
                                  xaxis_title="x (radial)", yaxis_title="y", zaxis_title="z (pin)")})
fig.update_layout(height=620, width=1200,
                  title=f"Carved fan — {name}  |  {FOLD_CLEARANCE_M*1e3:.2f} mm deck  (drag to rotate)")
fig.show()
